# Telco Customer Churn: Exploratory Data Analysis and Deep Learning
**Course:** Exploratory Data Analysis and Visualization  
**Project type:** Real-world dataset case study  
**Objective:** Explore customer churn patterns and train a neural network to predict whether a customer will leave a telecommunications provider.

> **Instructions:** Keep `Telco-Customer-Churn.csv` in the same folder as this notebook. In Google Colab, upload the CSV when prompted by the data-loading cell. Run cells from top to bottom. Interpret the actual results produced by your run; do not copy made-up metrics into the report.

## 1. Problem Statement and Objectives
Customer churn occurs when a customer stops using a company's services. High churn can affect revenue and customer-retention planning. This case study uses customer demographics, subscribed services, contract information, tenure and billing details to:
1. inspect and clean the dataset;
2. perform univariate, bivariate and multivariate EDA;
3. create at least five informative visualizations;
4. train and evaluate a Deep Learning Multi-Layer Perceptron (MLP) classifier; and
5. summarize findings, limitations and possible future improvements.

## 2. Import Libraries
The following libraries support data manipulation, visualization, preprocessing, model training and evaluation.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="deep")
print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)

## 3. Load the Dataset
The dataset is expected to contain customer records and a `Churn` target column. The code first looks for the CSV beside this notebook; if not found, it opens a file-upload prompt for Google Colab.

In [ ]:
DATA_PATH = "Telco-Customer-Churn.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "Place Telco-Customer-Churn.csv in the same folder as this notebook, "
            "or update DATA_PATH to the CSV location."
        )

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
display(df.head())

## 4. Understand the Dataset
Inspect the shape, column names, data types, descriptive statistics and target classes before making cleaning decisions.

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nNumerical summary:")
display(df.describe().T)

print("\nCategorical summary:")
display(df.describe(include="object").T)

print("\nChurn class counts:")
display(df["Churn"].value_counts(dropna=False))

## 5. Data Quality Checks
Check missing values, duplicate rows and duplicate customer identifiers. `TotalCharges` is commonly stored as text in this dataset, so convert it to numeric after trimming whitespace. Blank values become missing values and are handled in preprocessing.

In [ ]:
print("Missing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())
if "customerID" in df.columns:
    print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

df = df.drop_duplicates().copy()

# Convert TotalCharges to numeric; blank strings become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")

print("\nMissing values after TotalCharges conversion:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Updated dtypes:")
display(df.dtypes.astype(str).to_frame("dtype"))

## 6. Initial EDA: Target Distribution
The target distribution shows the number and proportion of customers who churned versus those who stayed. Class imbalance matters when interpreting accuracy.

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_percent = df["Churn"].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({"count": churn_counts, "percentage": churn_percent}))

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="Churn", order=["No", "Yes"])
ax.set(title="Customer Churn Distribution", xlabel="Churn", ylabel="Number of Customers")
for bar in ax.patches:
    ax.annotate(f"{int(bar.get_height()):,}", (bar.get_x()+bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")
plt.tight_layout()
plt.show()

## 7. Univariate Analysis: Tenure Distribution
A histogram helps examine how long customers have stayed with the company and whether tenure values cluster at particular durations.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="tenure", bins=30, kde=True)
plt.title("Distribution of Customer Tenure")
plt.xlabel("Tenure (months)")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

display(df["tenure"].describe().to_frame("tenure_statistics"))

## 8. Univariate Analysis: Monthly Charges
A histogram shows the distribution and spread of monthly charges. Skewness and unusual values can be considered during interpretation.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="MonthlyCharges", bins=30, kde=True)
plt.title("Distribution of Monthly Charges")
plt.xlabel("Monthly Charges")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

display(df["MonthlyCharges"].describe().to_frame("monthly_charges_statistics"))

## 9. Bivariate Analysis: Monthly Charges by Churn
Compare the distribution of monthly charges for customers who stayed and customers who left. The boxplot highlights medians, spread and potential outliers.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", order=["No", "Yes"])
plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.show()

## 10. Bivariate Analysis: Contract Type and Churn
A normalized stacked bar chart compares the proportion of churn outcomes within each contract type. This is more informative than raw counts alone when group sizes differ.

In [ ]:
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
display(contract_churn.round(2))

ax = contract_churn.reindex(columns=["No", "Yes"]).plot(
    kind="bar", stacked=True, figsize=(9, 5), ylim=(0, 100)
)
ax.set_title("Churn Percentage by Contract Type")
ax.set_xlabel("Contract Type")
ax.set_ylabel("Customers (%)")
ax.legend(title="Churn")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Bivariate Analysis: Tenure and Churn
This boxplot helps assess whether customers who leave tend to have shorter or longer tenure.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="tenure", order=["No", "Yes"])
plt.title("Customer Tenure by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Tenure (months)")
plt.tight_layout()
plt.show()

## 12. Multivariate Analysis: Numerical Correlation Heatmap
Correlation is calculated only for numerical columns. It describes linear associations, not causation. The binary `SeniorCitizen` field is numeric but represents a category.

In [ ]:
numeric_for_corr = df[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]].copy()
plt.figure(figsize=(7, 5))
sns.heatmap(numeric_for_corr.corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Heatmap of Selected Numerical Features")
plt.tight_layout()
plt.show()

## 13. Additional Insight: Churn by Internet Service
Compare churn rates across internet-service categories. The chart reports percentages within each category, not just total customers.

In [ ]:
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
display(internet_churn.round(2))

ax = internet_churn.reindex(columns=["No", "Yes"]).plot(
    kind="bar", stacked=True, figsize=(8, 5), ylim=(0, 100)
)
ax.set_title("Churn Percentage by Internet Service")
ax.set_xlabel("Internet Service")
ax.set_ylabel("Customers (%)")
ax.legend(title="Churn")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 14. Outlier Review and Cleaning Decisions
Boxplots can flag unusually high or low numerical observations, but an outlier is not automatically an error. Customer charges and tenure can legitimately vary. For this case study, retain plausible values and use scaling for numerical features. The model pipeline imputes missing numerical values using the median and categorical values using the most frequent category.

In [ ]:
numeric_cols_review = ["tenure", "MonthlyCharges", "TotalCharges"]
outlier_summary = []
for col in numeric_cols_review:
    series = df[col].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((series < lower) | (series > upper)).sum())
    outlier_summary.append({
        "feature": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_bound": lower, "upper_bound": upper,
        "flagged_count": count
    })
display(pd.DataFrame(outlier_summary).round(2))

## 15. Prepare Features and Target
- `Churn` is the target: `Yes` = 1 and `No` = 0.
- `customerID` is an identifier, not a useful predictive feature, so it is excluded.
- Categorical features are one-hot encoded.
- Numerical features are median-imputed and standardized.
- Data is split into training, validation and test sets using stratification to preserve the target proportions.
- The preprocessing pipeline is fitted on the training set only to avoid data leakage.

In [ ]:
model_df = df.dropna(subset=["Churn"]).copy()
y = model_df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
X = model_df.drop(columns=["Churn", "customerID"], errors="ignore")

# Split: 70% train, 15% validation, 15% test.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Convert sparse matrices to dense arrays for TensorFlow.
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_val_processed = X_val_processed.toarray()
    X_test_processed = X_test_processed.toarray()

X_train_processed = X_train_processed.astype("float32")
X_val_processed = X_val_processed.astype("float32")
X_test_processed = X_test_processed.astype("float32")
y_train_np = y_train.to_numpy(dtype="float32")
y_val_np = y_val.to_numpy(dtype="float32")
y_test_np = y_test.to_numpy(dtype="int32")

print("Training set:", X_train_processed.shape, y_train_np.shape)
print("Validation set:", X_val_processed.shape, y_val_np.shape)
print("Test set:", X_test_processed.shape, y_test_np.shape)
print("Numeric features:", numeric_features)
print("Categorical features:", len(categorical_features))

## 16. Deep Learning Model: MLP Classifier
A Multi-Layer Perceptron is suitable for this tabular binary-classification task. The model uses two hidden dense layers with ReLU activation, dropout for regularization and a sigmoid output for churn probability. Early stopping monitors validation loss and restores the best weights.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_processed.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc")
    ]
)

model.summary()

## 17. Train the Model
Training parameters: Adam optimizer, learning rate 0.001, binary cross-entropy loss, batch size 32, maximum 100 epochs, and early stopping based on validation loss.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = model.fit(
    X_train_processed, y_train_np,
    validation_data=(X_val_processed, y_val_np),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)
print("Training stopped after", len(history.history["loss"]), "epochs.")

## 18. Model Evaluation: Loss vs Epoch
Training and validation loss help show whether the model is learning and whether the validation performance begins to worsen.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 19. Model Evaluation: Accuracy vs Epoch
Compare training and validation accuracy. A large gap can indicate overfitting; interpret the curve alongside loss and test metrics.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## 20. Test Set Predictions and Metrics
The final evaluation uses the held-out test set. The default classification threshold is 0.50. Because churn classes may be imbalanced, review precision, recall, F1-score and ROC-AUC alongside accuracy.

In [ ]:
y_probability = model.predict(X_test_processed, verbose=0).ravel()
y_pred = (y_probability >= 0.50).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test_np, y_pred),
    "Precision": precision_score(y_test_np, y_pred, zero_division=0),
    "Recall": recall_score(y_test_np, y_pred, zero_division=0),
    "F1-score": f1_score(y_test_np, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test_np, y_probability)
}
metrics_df = pd.DataFrame({"Metric": list(metrics.keys()), "Score": list(metrics.values())})
display(metrics_df.round(4))
print("\nClassification report:")
print(classification_report(y_test_np, y_pred, target_names=["No churn", "Churn"], zero_division=0))

## 21. Model Evaluation: Confusion Matrix
The confusion matrix shows true negatives, false positives, false negatives and true positives. False negatives are customers who churn but were predicted to stay.

In [ ]:
cm = confusion_matrix(y_test_np, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No churn", "Churn"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix on Test Set")
plt.tight_layout()
plt.show()

## 22. Model Evaluation: ROC Curve
The ROC curve shows the trade-off between true-positive rate and false-positive rate across classification thresholds. AUC closer to 1 indicates better ranking ability; 0.5 is approximately random ranking.

In [ ]:
RocCurveDisplay.from_predictions(y_test_np, y_probability)
plt.title(f"ROC Curve (AUC = {roc_auc_score(y_test_np, y_probability):.3f})")
plt.tight_layout()
plt.show()

## 23. Model Evaluation: Error / Probability Distribution
Compare predicted churn probabilities for customers who actually stayed and customers who churned. Stronger separation suggests the model distinguishes the classes more effectively.

In [ ]:
prob_df = pd.DataFrame({
    "Actual churn": np.where(y_test_np == 1, "Churn", "No churn"),
    "Predicted churn probability": y_probability
})
plt.figure(figsize=(8, 5))
sns.histplot(
    data=prob_df, x="Predicted churn probability",
    hue="Actual churn", bins=25, stat="density",
    common_norm=False, element="step", fill=False
)
plt.title("Predicted Churn Probability by Actual Class")
plt.xlabel("Predicted probability of churn")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

## 24. Feature Analysis
The model uses the available customer attributes after preprocessing. The next table lists the transformed feature names. For a neural network, coefficients do not provide the same direct feature-importance interpretation as a linear model or tree model; avoid claiming definitive feature importance from the MLP alone.

In [ ]:
try:
    transformed_feature_names = preprocessor.get_feature_names_out()
    print("Number of transformed features:", len(transformed_feature_names))
    display(pd.DataFrame({"transformed_feature": transformed_feature_names}).head(50))
except Exception as e:
    print("Feature names could not be displayed:", e)

## 25. Key Findings Template
Use the actual charts and outputs above to complete this section. Do not write a conclusion until you have run the notebook and checked the results.

- **Churn distribution:** State the observed number and percentage of churned customers.
- **Tenure:** Describe the main pattern in the tenure histogram and tenure-by-churn boxplot.
- **Monthly charges:** Compare the medians and spreads for churned and retained customers.
- **Contract type:** Identify which contract category has the highest observed churn percentage.
- **Internet service:** Compare the churn percentages among internet-service categories.
- **Correlation:** Report the strongest visible numerical associations, noting that correlation does not imply causation.
- **Model:** Report test accuracy, precision, recall, F1-score and ROC-AUC from the metrics table. Mention the confusion-matrix errors and whether training curves suggest overfitting.

## 26. Conclusion and Future Scope
### Conclusion
This case study applies exploratory data analysis, data cleaning, statistical summaries and visualization to a real-world telecommunications customer dataset. A deep-learning MLP classifier is trained to estimate customer churn. Insert the specific patterns and actual test metrics from your notebook outputs before submission.

### Future Scope
1. Compare the MLP with Logistic Regression, Random Forest and Gradient Boosting.
2. Tune the decision threshold according to the business cost of missing a churner versus contacting a customer who would stay.
3. Use cross-validation and systematic hyperparameter tuning.
4. Apply explainability methods such as permutation importance or SHAP to investigate feature contributions.
5. Evaluate on newer customer data to assess whether performance remains stable over time.

### Limitations
The dataset is a snapshot, observed relationships do not establish causation, the target may be imbalanced, and model performance depends on the available features and chosen threshold. The model's predictions should support—not replace—business judgment.

## 27. Literature Review and References
Use at least 3–5 references, formatted consistently in APA or IEEE style. The following are relevant starting points; verify bibliographic details and access dates according to your institution's preferred style.

1. IBM, *Telco Customer Churn* dataset and sample project materials. GitHub: https://github.com/IBM/telco-customer-churn-on-icp4d
2. TensorFlow, *Keras Sequential model guide*. https://www.tensorflow.org/guide/keras/sequential_model
3. scikit-learn developers, *ColumnTransformer documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
4. scikit-learn developers, *Model evaluation: scoring, metrics and evaluation*. https://scikit-learn.org/stable/modules/model_evaluation.html
5. McKinney, W. *Python for Data Analysis*. O'Reilly Media. (Use the edition available to you and format the year/edition correctly.)

Add at least one research paper about customer churn prediction from a library such as Google Scholar or your institution's database, and cite it in the report if you discuss its findings.

## 28. Reproducibility and Save Outputs
This cell saves the test metrics and a compact CSV of actual labels, predicted labels and predicted probabilities. Keep these outputs as evidence for the report appendix if needed.

In [ ]:
metrics_df.to_csv("model_evaluation_metrics.csv", index=False)
predictions_df = pd.DataFrame({
    "actual_churn": y_test_np,
    "predicted_churn": y_pred,
    "predicted_churn_probability": y_probability
})
predictions_df.to_csv("test_predictions.csv", index=False)
print("Saved: model_evaluation_metrics.csv and test_predictions.csv")

## 29. GitHub Submission Checklist
Before submission, ensure the public repository contains:
- [ ] Project report in PDF or Word format
- [ ] Completed notebook (`.ipynb`) with outputs
- [ ] Dataset file or a clear dataset download link
- [ ] `README.md` with the problem, dataset source, setup instructions, methods and key findings
- [ ] Any saved evaluation outputs you want to include
- [ ] No fabricated metrics; report the results from your actual run

Suggested repository name: `telco-customer-churn-eda-deep-learning`